In [ ]:
import requests
import pandas as pd
import time

def fetch_steam_reviews(appid: str, game_name: str, max_reviews: int = 1500, language: str = "english") -> pd.DataFrame:
    url = f"https://store.steampowered.com/appreviews/{appid}"
    cursor = "*"
    rows = []

    while len(rows) < max_reviews:
        params = {
            "json": 1,
            "cursor": cursor,
            "num_per_page": 100,
            "language": language,
            "filter": "updated",
            "purchase_type": "all"
        }

        try:
            response = requests.get(url, params=params, timeout=30)
            response.raise_for_status()
            data = response.json()
        except Exception as e:
            print(f"Request failed: {e}")
            break

        reviews = data.get("reviews", [])
        if not reviews:
            print("No more reviews returned.")
            break

        for r in reviews:
            rows.append({
                "game": game_name,
                "platform": "Steam",
                "review_id": r.get("recommendationid"),
                "review_text": r.get("review"),
                "voted_up": r.get("voted_up"),
                "votes_up": r.get("votes_up"),
                "votes_funny": r.get("votes_funny"),
                "weighted_vote_score": r.get("weighted_vote_score"),
                "comment_count": r.get("comment_count"),
                "steam_purchase": r.get("steam_purchase"),
                "received_for_free": r.get("received_for_free"),
                "written_during_early_access": r.get("written_during_early_access"),
                "timestamp_created": r.get("timestamp_created"),
                "timestamp_updated": r.get("timestamp_updated"),
                "playtime_forever": r.get("author", {}).get("playtime_forever"),
                "playtime_last_two_weeks": r.get("author", {}).get("playtime_last_two_weeks"),
                "playtime_at_review": r.get("author", {}).get("playtime_at_review"),
                "last_played": r.get("author", {}).get("last_played"),
            })

            if len(rows) >= max_reviews:
                break

        cursor = data.get("cursor", cursor)
        print(f"Fetched {len(rows)} reviews so far...")
        time.sleep(1)

    df = pd.DataFrame(rows)

    if not df.empty:
        df["timestamp_created"] = pd.to_datetime(df["timestamp_created"], unit="s", errors="coerce")
        df["timestamp_updated"] = pd.to_datetime(df["timestamp_updated"], unit="s", errors="coerce")
        df["last_played"] = pd.to_datetime(df["last_played"], unit="s", errors="coerce")

        df["playtime_hours"] = df["playtime_forever"] / 60
        df["playtime_at_review_hours"] = df["playtime_at_review"] / 60
        df["review_length"] = df["review_text"].astype(str).str.len()

        df = df.drop_duplicates(subset=["review_id"])
        df = df.dropna(subset=["review_text"])
        df["review_text"] = df["review_text"].astype(str).str.strip()
        df = df[df["review_text"] != ""]

    return df

APEX_APPID = "1172470"

df_apex = fetch_steam_reviews(
    appid=APEX_APPID,
    game_name="Apex Legends",
    max_reviews=1500,
    language="english"
)

print("\nFinished!")
print(df_apex.shape)
print(df_apex.head())

df_apex.to_csv("apex_steam_reviews_english_1500.csv", index=False, encoding="utf-8-sig")
print("Saved to apex_steam_reviews_english_1500.csv")

In [ ]:
import re

def is_mostly_english(text, threshold=0.8):
    text = str(text)
    if not text.strip():
        return False
    english_chars = re.findall(r"[A-Za-z\s]", text)
    ratio = len(english_chars) / max(len(text), 1)
    return ratio >= threshold

df_apex["is_english"] = df_apex["review_text"].apply(is_mostly_english)
print(df_apex["is_english"].value_counts())

df_apex_en = df_apex[df_apex["is_english"] == True].copy()
print(df_apex_en.shape)

df_apex_en.to_csv("apex_steam_reviews_english_filtered.csv", index=False, encoding="utf-8-sig")
print("Saved filtered Apex CSV")

In [ ]:
df_apex = df_apex_en.copy()

In [ ]:
print(df_apex[["review_id", "review_text", "voted_up", "playtime_hours"]].head())
print(df_apex["voted_up"].value_counts(normalize=True))
print(df_apex["review_length"].describe())

print(df_apex.groupby("voted_up")["playtime_hours"].describe())
print(df_apex.groupby("voted_up")["review_length"].describe())

In [ ]:
import re
from collections import Counter

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_apex["clean_text"] = df_apex["review_text"].apply(clean_text)

custom_stopwords_apex = {
    "the","a","an","and","or","but","if","then","so","of","to","in","on","for","with","at","by","from",
    "is","it","this","that","these","those","be","been","being","am","are","was","were",
    "i","me","my","mine","you","your","yours","he","she","they","them","their","we","our","us",
    "as","not","no","yes","do","does","did","have","has","had","can","could","would","should","shall",

    "good","great","best","bad","nice","okay","ok","love","like","liked",
    "really","very","pretty","quite","much","many","some","any","all","every","more","most","less",
    "still","even","just","also","well","too","only","there","here","what","when","where","why","how",

    "im","ive","id","youre","dont","didnt","cant","couldnt","wont","wasnt","isnt","arent","don","its",

    "game","games","play","played","playing","player","players","hours","hour","time",
    "one","two","first","second","get","got","go","going","new","old","make","made","say","said",
    "will","would","way","thing","things","something","anything","everything","someone","anyone",

    "apex","legends"
}

def tokenize_and_filter(text):
    tokens = text.split()
    tokens = [w for w in tokens if w not in custom_stopwords_apex and len(w) >= 3]
    return tokens

df_apex["tokens"] = df_apex["clean_text"].apply(tokenize_and_filter)

In [ ]:
extra_stopwords_apex = {
    "better", "out", "back", "now", "other", "ever", "lot", "than",
    "after", "because", "getting", "into", "people", "off", "who",
    "want", "never"
}

custom_stopwords_apex = custom_stopwords_apex.union(extra_stopwords_apex)

In [ ]:
extra_stopwords_apex_2 = {
    "lock", "about", "sometimes", "keep", "actually", "years",
    "overall", "need", "over", "again", "full", "instead"
}

custom_stopwords_apex = custom_stopwords_apex.union(extra_stopwords_apex_2)

In [ ]:
extra_stopwords_apex_3 = {"hate", "worst", "trash", "ass"}
custom_stopwords_apex = custom_stopwords_apex.union(extra_stopwords_apex_3)

In [ ]:
extra_stopwords_apex_4 = {
    "cool", "life", "feels", "though", "always", "know",
    "once", "match", "season", "linux"
}

custom_stopwords_apex = custom_stopwords_apex.union(extra_stopwords_apex_4)

In [ ]:
def tokenize_and_filter(text):
    tokens = text.split()
    tokens = [w for w in tokens if w not in custom_stopwords_apex and len(w) >= 4]
    return tokens

df_apex["tokens"] = df_apex["clean_text"].apply(tokenize_and_filter)

In [ ]:
from collections import Counter

def get_top_words_from_tokens(token_series, top_n=20):
    counter = Counter()
    for tokens in token_series.dropna():
        counter.update(tokens)
    return counter.most_common(top_n)

positive_words_apex = get_top_words_from_tokens(
    df_apex[df_apex["voted_up"] == True]["tokens"], top_n=20
)

negative_words_apex = get_top_words_from_tokens(
    df_apex[df_apex["voted_up"] == False]["tokens"], top_n=20
)

print("Top positive words (Apex):")
print(positive_words_apex)

print("\nTop negative words (Apex):")
print(negative_words_apex)

In [ ]:
lock_positive = df_apex[
    (df_apex["voted_up"] == True) &
    (df_apex["clean_text"].str.contains(r"\block\b", na=False))
][["review_text"]]

lock_negative = df_apex[
    (df_apex["voted_up"] == False) &
    (df_apex["clean_text"].str.contains(r"\block\b", na=False))
][["review_text"]]

print("Positive lock mentions:")
print(lock_positive.head(20))

print("\nNegative lock mentions:")
print(lock_negative.head(20))

In [ ]:
apex_theme_keywords = {
    # 原有全部保留
    "fun", "movement", "battle", "royale", "friends", "skill",
    "gameplay", "gun", "fps", "shooter", "cheaters", "ranked",
    "servers", "update", "respawn", "fix", "system", "devs",
    "skins", "money",
    # 新加的
    "matchmaking", "sbmm", "ban", "ping", "ea", "hacker",
    "aimassist", "ttk", "lag", "crash"
}

In [ ]:
def get_keyword_counts(token_series, keyword_set):
    counter = Counter()
    for tokens in token_series.dropna():
        for t in tokens:
            if t in keyword_set:
                counter[t] += 1
    return counter

positive_keyword_counts_apex = get_keyword_counts(
    df_apex[df_apex["voted_up"] == True]["tokens"],
    apex_theme_keywords
)

negative_keyword_counts_apex = get_keyword_counts(
    df_apex[df_apex["voted_up"] == False]["tokens"],
    apex_theme_keywords
)

print("Positive keyword counts (Apex):")
print(positive_keyword_counts_apex.most_common())

print("\nNegative keyword counts (Apex):")
print(negative_keyword_counts_apex.most_common())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

focus_keywords_apex = [
    "movement", "battle", "royale", "friends", "skill", "gameplay",
    "cheaters", "ranked", "servers", "update", "money", "skins",
    # 新加的
    "matchmaking", "ban", "ea", "ping", "lag"
]

pos_dict_apex = dict(positive_keyword_counts_apex)
neg_dict_apex = dict(negative_keyword_counts_apex)

compare_df_apex = pd.DataFrame({
    "keyword": focus_keywords_apex,
    "positive": [pos_dict_apex.get(k, 0) for k in focus_keywords_apex],
    "negative": [neg_dict_apex.get(k, 0) for k in focus_keywords_apex]
})

print(compare_df_apex)

compare_df_apex = compare_df_apex.sort_values("positive", ascending=True)

y = np.arange(len(compare_df_apex))
height = 0.38

plt.figure(figsize=(10, 6))
plt.barh(y - height/2, compare_df_apex["positive"], height=height, label="Positive Reviews")
plt.barh(y + height/2, compare_df_apex["negative"], height=height, label="Negative Reviews")

plt.yticks(y, compare_df_apex["keyword"])
plt.xlabel("Count")
plt.ylabel("Keyword")
plt.title("Apex Legends: Positive vs Negative Review Themes")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
short_negative_apex = df_apex[
    (df_apex["playtime_hours"] < 10) &
    (df_apex["voted_up"] == False)
][["review_text", "playtime_hours"]].sort_values("playtime_hours", ascending=False)

print(short_negative_apex.head(30))

In [ ]:
import pandas as pd
import numpy as np

bins = [0, 10, 50, 100, 300, np.inf]
labels = ["0–10h", "10–50h", "50–100h", "100–300h", "300h+"]

df_apex["playtime_bin"] = pd.cut(
    df_apex["playtime_hours"],
    bins=bins,
    labels=labels,
    right=False
)

print(df_apex[["playtime_hours", "playtime_bin"]].head())

In [ ]:
mid_long_negative_apex = df_apex[
    (df_apex["playtime_bin"] == "100–300h") &
    (df_apex["voted_up"] == False)
][["review_text", "playtime_hours"]].sort_values("playtime_hours", ascending=False)

print(mid_long_negative_apex.head(30))

In [ ]:
ultra_long_negative_apex = df_apex[
    (df_apex["playtime_bin"] == "300h+") &
    (df_apex["voted_up"] == False)
][["review_text", "playtime_hours"]].sort_values("playtime_hours", ascending=False)

print(ultra_long_negative_apex.head(30))

In [ ]:
def classify_short_negative_apex(text):
    t = str(text).lower()

    if any(k in t for k in ["cheater", "cheaters", "hacker", "hackers"]):
        return "fairness/cheating"
    elif any(k in t for k in ["server", "lag", "stutter", "crash", "unplayable", "fps"]):
        return "technical/performance"
    elif any(k in t for k in ["matchmaking", "sbmm", "ranked", "teammates"]):
        return "matchmaking/competitive friction"
    elif any(k in t for k in ["money", "skin", "skins", "pay", "cash", "microtransaction"]):
        return "monetisation"
    elif any(k in t for k in ["lock", "ban", "banned", "anti cheat", "eac"]):
        return "account/anti-cheat issue"
    elif len(t.strip()) <= 15:
        return "low-information"
    else:
        return "other"

In [ ]:
short_negative_apex["early_negative_reason"] = short_negative_apex["review_text"].apply(classify_short_negative_apex)
print(short_negative_apex["early_negative_reason"].value_counts())

In [ ]:
def classify_long_negative_apex(text):
    t = str(text).lower()

    if any(k in t for k in ["cheater", "cheaters", "hacker", "hackers"]):
        return "fairness/cheating"
    elif any(k in t for k in ["ranked", "matchmaking", "sbmm"]):
        return "ranked/matchmaking"
    elif any(k in t for k in ["server", "lag", "stutter", "crash", "unplayable"]):
        return "technical/performance"
    elif any(k in t for k in ["update", "patch", "respawn", "devs", "developer"]):
        return "update/dev dissatisfaction"
    elif any(k in t for k in ["money", "skin", "skins", "microtransaction", "store"]):
        return "monetisation"
    elif len(t.strip()) <= 15:
        return "low-information"
    else:
        return "other"

In [ ]:
import pandas as pd
import numpy as np

bins = [0, 10, 50, 100, 300, 1000, np.inf]
labels = ["0–10h", "10–50h", "50–100h", "100–300h", "300–1000h", "1000h+"]

df_apex["playtime_bin"] = pd.cut(
    df_apex["playtime_hours"],
    bins=bins,
    labels=labels,
    right=False
)

print(df_apex[["playtime_hours", "playtime_bin"]].head())

In [ ]:
apex_bin_summary = (
    df_apex.groupby("playtime_bin")
    .agg(
        review_count=("voted_up", "count"),
        positive_rate=("voted_up", "mean"),
        median_hours=("playtime_hours", "median")
    )
    .reset_index()
)

print(apex_bin_summary)

In [ ]:
apex_sentiment_bin = (
    df_apex.groupby(["playtime_bin", "voted_up"])
    .size()
    .unstack(fill_value=0)
)

apex_sentiment_bin_pct = apex_sentiment_bin.div(
    apex_sentiment_bin.sum(axis=1), axis=0
)

print(apex_sentiment_bin_pct)

In [ ]:
import matplotlib.pyplot as plt

apex_sentiment_bin_pct.plot(
    kind="bar",
    stacked=True,
    figsize=(9, 5.5)
)

plt.xlabel("Playtime Group")
plt.ylabel("Share of Reviews")
plt.title("Apex Legends: Review Outcome Composition by Playtime Group")
plt.legend(title="Recommended", labels=["Negative", "Positive"])
plt.tight_layout()
plt.show()

In [ ]:
apex_short_negative = df_apex[
    (df_apex["playtime_bin"] == "0–10h") &
    (df_apex["voted_up"] == False)
][["review_text", "playtime_hours"]].sort_values("playtime_hours", ascending=False)

print(apex_short_negative.head(30))

In [ ]:
apex_midlong_negative = df_apex[
    (df_apex["playtime_bin"] == "300–1000h") &
    (df_apex["voted_up"] == False)
][["review_text", "playtime_hours"]].sort_values("playtime_hours", ascending=False)

print(apex_midlong_negative.head(30))

In [ ]:
apex_ultralong_negative = df_apex[
    (df_apex["playtime_bin"] == "1000h+") &
    (df_apex["voted_up"] == False)
][["review_text", "playtime_hours"]].sort_values("playtime_hours", ascending=False)

print(apex_ultralong_negative.head(30))

In [ ]:
print("0–10h negative:", len(apex_short_negative))
print("300–1000h negative:", len(apex_midlong_negative))
print("1000h+ negative:", len(apex_ultralong_negative))

In [ ]:
import pandas as pd

# 导出正评关键词频率
pos_export_apex = pd.DataFrame(
    positive_keyword_counts_apex.most_common(),
    columns=["keyword", "positive_count"]
)

# 导出负评关键词频率
neg_export_apex = pd.DataFrame(
    negative_keyword_counts_apex.most_common(),
    columns=["keyword", "negative_count"]
)

# 合并
keyword_export_apex = pos_export_apex.merge(neg_export_apex, on="keyword", how="outer").fillna(0)
keyword_export_apex["positive_count"] = keyword_export_apex["positive_count"].astype(int)
keyword_export_apex["negative_count"] = keyword_export_apex["negative_count"].astype(int)

# 保存
keyword_export_apex.to_csv("apex_keyword_counts.csv", index=False, encoding="utf-8-sig")
print("已保存！")
print(keyword_export_apex)

In [ ]:
pos_export_apex = pd.DataFrame(
    positive_keyword_counts_apex.most_common(),
    columns=["keyword", "positive_count"]
)
neg_export_apex = pd.DataFrame(
    negative_keyword_counts_apex.most_common(),
    columns=["keyword", "negative_count"]
)
keyword_export_apex = pos_export_apex.merge(neg_export_apex, on="keyword", how="outer").fillna(0)
keyword_export_apex["positive_count"] = keyword_export_apex["positive_count"].astype(int)
keyword_export_apex["negative_count"] = keyword_export_apex["negative_count"].astype(int)
keyword_export_apex.to_csv("apex_keyword_counts.csv", index=False, encoding="utf-8-sig")
print("Apex导出完成！")